In [8]:
# 数据清洗
import os
import scipy.io
import pandas as pd
import numpy as np
import re
import sys
from pathlib import Path
notebook_path = Path(sys.argv[0]).parent if "ipykernel" in sys.argv[0] else Path.cwd()
sys.path.append(str(notebook_path))
from utils import use  

def extract_load(filename):
    """从文件名中提取载荷信息（下划线后面的数字）"""
    if '_' in filename:
        return filename.split('_')[-1].split('.')[0]
    return None

def extract_sampling_position(folder_name):
    """从文件夹名中提取轴承位置（如从12kHz_DE_data提取DE）"""
    parts = folder_name.split('_')
    if len(parts) >= 2 and parts[-1] == 'data':
        return parts[-2]
    return None

def extract_sampling_freq(folder_name):
    """从文件夹名中提取采样频率（如从12kHz_DE_data提取12）"""
    if 'kHz' in folder_name:
        return folder_name.split('kHz')[0]
    return None

def extract_position2(key):
    """从数据键中提取采样位置（如从X119_DE_time提取DE）"""
    parts = key.split('_')
    if len(parts) >= 2 and parts[-1] == 'time':
        return parts[-2]
    return None

def process_mat_file(file_path, sampling_freq, fault_type, fault_size, 
                     sampling_pos, or_sampling_pos=None):
    """处理单个mat文件，返回要添加到DataFrame的行数据"""
    try:
        # 读取mat文件
        mat_data = scipy.io.loadmat(file_path)
        
        # 提取文件名和载荷
        file_name = os.path.basename(file_path)
        load = extract_load(file_name)
        
        # 提取RPM值
        rpm_keys = [k for k in mat_data.keys() if 'RPM' in k]
        rpm = mat_data[rpm_keys[0]][0][0] if rpm_keys else None
        
        # 查找所有时间序列数据
        time_keys = [k for k in mat_data.keys() if '_time' in k and not k.startswith('__')]

        #提取编号
        number = rpm_keys[0][:-3]  if rpm_keys else None# 去掉末尾"RPM"得到编号
        
        rows = []
        for key in time_keys:
            # 提取数据
            data = mat_data[key].flatten()  # 转换为一维数组
            
            # 提取采样位置
            pos2 = extract_position2(key)
            
            # 创建行数据
            row = {
                '数据': data,
                '采样频率': sampling_freq,
                '故障类型': fault_type,
                '故障尺寸': fault_size,
                '轴承位置': sampling_pos,
                '采样位置': pos2,
                'rpm': rpm,
                '载荷': load,
                '文件名': file_name,
                'or采样位置': or_sampling_pos,
                '编号':number
            }
            rows.append(row)
        
        return rows
    
    except Exception as e:
        print(f"处理文件 {file_path} 时出错: {str(e)}")
        return []

def create_dataset(root_dir):
    """创建数据集主函数"""
    # 创建空白DataFrame
    df = pd.DataFrame(columns=[
        '数据', '采样频率', '故障类型', '故障尺寸', 
        '轴承位置', '采样位置', 'rpm', '载荷', 
        '文件名', 'or采样位置'
    ])
    
    # 要遍历的第一层子文件夹
    first_level_folders = ['12kHz_FE_data', '12kHz_DE_data', '48kHz_DE_data']
    
    for first_folder in first_level_folders:
        first_path = os.path.join(root_dir, first_folder)
        if not os.path.exists(first_path):
            print(f"文件夹 {first_path} 不存在，跳过")
            continue
        
        # 提取采样频率和采样位置
        sampling_freq = extract_sampling_freq(first_folder)
        sampling_pos = extract_sampling_position(first_folder)
        
        # 第二层文件夹：故障类型
        second_level_folders = ['B', 'IR', 'OR']
        for second_folder in second_level_folders:
            second_path = os.path.join(first_path, second_folder)
            if not os.path.exists(second_path):
                print(f"文件夹 {second_path} 不存在，跳过")
                continue
            
            fault_type = second_folder  # 故障类型
            
            # 处理OR的特殊情况（多一层文件夹）
            if second_folder == 'OR':
                # OR文件夹下的额外层级
                or_pos_folders = ['Centered', 'Opposite', 'Orthogonal']
                for or_pos_folder in or_pos_folders:
                    or_pos_path = os.path.join(second_path, or_pos_folder)
                    if not os.path.exists(or_pos_path):
                        print(f"文件夹 {or_pos_path} 不存在，跳过")
                        continue
                    
                    # 第三层文件夹：故障尺寸
                    for third_folder in os.listdir(or_pos_path):
                        third_path = os.path.join(or_pos_path, third_folder)
                        if not os.path.isdir(third_path):
                            continue
                        
                        fault_size = third_folder  # 故障尺寸
                        
                        # 处理该文件夹下的所有mat文件（跳过隐藏文件）
                        # 处理mat文件（跳过隐藏文件）
                        for file in os.listdir(third_path):
                            if file.startswith('.') or not file.endswith('.mat'):
                                continue  # 跳过隐藏文件和非mat文件
                            file_path = os.path.join(third_path, file)
                            rows = process_mat_file(
                                file_path, sampling_freq, second_folder, 
                                third_folder, sampling_pos, or_pos_folder
                            )
                            if rows:
                                df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True)
            else:
                # 处理B和IR的情况
                for third_folder in os.listdir(second_path):
                    third_path = os.path.join(second_path, third_folder)
                    if not os.path.isdir(third_path):
                        continue
                    
                    for file in os.listdir(third_path):
                        if file.startswith('.') or not file.endswith('.mat'):
                            continue  # 跳过隐藏文件
                        file_path = os.path.join(third_path, file)
                        rows = process_mat_file(
                            file_path, sampling_freq, second_folder, 
                            third_folder, sampling_pos
                        )
                        if rows:
                            df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True)
            
            #生成故障标签
            df['故障标签'] = df['文件名'].str.split('_').str[0] 
            df['故障标签'] = df['故障标签'].str.split('@').str[0]

    return df

#读入正常数据
def process_normal_data(normal_data_path, dataset):
    """
    处理正常数据并添加到现有数据集
    
    参数:
    normal_data_path: 正常数据文件夹路径
    dataset: 现有数据集
    """
    # 提取采样频率和采样位置
    folder_name = os.path.basename(normal_data_path)
    sampling_freq = folder_name.split('kHz')[0]  # 提取48
    sampling_pos = folder_name.split('_')[1]     # 提取Normal前的部分
    
    # 遍历文件夹中的所有mat文件
    for file in os.listdir(normal_data_path):
        if file.endswith('.mat'):
            if file.startswith('.') :
                continue  # 跳过隐藏文件
            file_path = os.path.join(normal_data_path, file)
            try:
                # 读取mat文件
                mat_data = scipy.io.loadmat(file_path)
                
                # 提取文件名和载荷（正常数据载荷设为0）
                file_name = os.path.basename(file_path)
                load = extract_load(file_name)
                
                # 提取RPM值
                rpm_keys = [k for k in mat_data.keys() if 'RPM' in k]
                rpm = mat_data[rpm_keys[0]][0][0] if rpm_keys else None

                #提取编号
                number = rpm_keys[0][:-3]  if rpm_keys else None# 去掉末尾"RPM"得到编号
                
                # 查找所有时间序列数据
                time_keys = [k for k in mat_data.keys() if '_time' in k and not k.startswith('__')]
                
                # 处理每个时间序列
                for key in time_keys:
                    data = mat_data[key].flatten()  # 转换为一维数组
                    pos2 = extract_position2(key)   # 提取采样位置2
                    
                    # 创建正常数据行
                    new_row = {
                        '数据': data,
                        '采样频率': sampling_freq,
                        '故障类型': 'Normal',  # 正常数据标记
                        '故障尺寸': None,     # 无故障尺寸
                        '轴承位置': sampling_pos,
                        '采样位置': pos2,
                        'rpm': rpm,
                        '载荷': load,
                        '文件名': file_name,
                        'or采样位置': None,   # 无OR采样位置
                        '故障标签': 'Normal'  # 故障标签设为Normal
                       ,'编号':number
                    }
                    
                    # 添加到数据集
                    dataset = pd.concat([dataset, pd.DataFrame([new_row])], ignore_index=True)
            
            except Exception as e:
                print(f"处理正常数据文件 {file_path} 时出错: {str(e)}")
    
    return dataset

#处理rpm为空的情况
def extract_rpm_from_filename(filename):
    # 使用正则表达式匹配文件名中的数字+rpm格式（如1797rpm）
    match = re.search(r'(\d+)rpm', filename)
    if match:
        # 提取数字部分并转换为整数
        return int(match.group(1))
    return None


In [9]:
#读入故障数据
source_dir = r'I:\Pycharm Projects\2025ys\数据集\源域数据集'
dataset = create_dataset(source_dir)

# 读入正常数据
normal_data_path = r"I:\Pycharm Projects\2025ys\数据集\源域数据集\48kHz_Normal_data"
updated_dataset = process_normal_data(normal_data_path, dataset)

#统计数据长度信息
updated_dataset['数据长度'] = updated_dataset['数据'].apply(lambda x: len(x) if isinstance(x, np.ndarray) else 0)

#处理rpm为空的情况
updated_dataset.loc[updated_dataset['rpm'].isna(), 'rpm'] = updated_dataset.loc[updated_dataset['rpm'].isna(), '文件名'].apply(extract_rpm_from_filename)

# 手动填补编号，根据index和已知的编号进行赋值
df = updated_dataset.copy()
df.loc[171, '编号'] = 'X048'
df.loc[172, '编号'] = 'X049'
df.loc[173, '编号'] = 'X049'
df.loc[174, '编号'] = 'X050'
df.loc[211, '编号'] = 'X056'
df.loc[212, '编号'] = 'X057'
df.loc[213, '编号'] = 'X058'
df.loc[214, '编号'] = 'X059'
df.loc[405, '编号'] = 'X098'
df.loc[406, '编号'] = 'X098'
df.loc[407, '编号'] = 'X099'
df.loc[408, '编号'] = 'X099'

#修改载荷信息
df.loc[pd.to_numeric(df['载荷'], errors='coerce').isna(), '载荷'] = df.loc[pd.to_numeric(df['载荷'], errors='coerce').isna(), '文件名'].apply(
    lambda x: x.split('_')[1] if len(x.split('_')) >=2 else '未知'  # 防止分割失败
)

# 打印数据集基本信息
print(f"数据集创建完成，共 {len(df)} 行数据")
print("数据集列名：", df.columns.tolist())

df.to_pickle('E:\故障诊断2\实验整理\dataset\processed\CWRU_bearing_dataset.pkl')

#确定此时df无空
df.isnull().sum()

数据集创建完成，共 411 行数据
数据集列名： ['数据', '采样频率', '故障类型', '故障尺寸', '轴承位置', '采样位置', 'rpm', '载荷', '文件名', 'or采样位置', '编号', '故障标签', '数据长度']


数据          0
采样频率        0
故障类型        0
故障尺寸        8
轴承位置        0
采样位置        0
rpm         0
载荷          0
文件名         0
or采样位置    208
编号          0
故障标签        0
数据长度        0
dtype: int64

In [1]:
import pandas as pd
df = pd.read_pickle(r'E:\故障诊断2\实验整理\dataset\processed\CWRU_bearing_dataset.pkl')


In [11]:
df['数据']

0      [-0.0619336764306459, -0.038543017748011166, -...
1      [0.013047125137835886, 0.04150167059238134, 0....
2      [0.12537047425712516, -0.04568046540657417, -0...
3      [-0.023941737485286378, -0.03157618858309077, ...
4      [-0.07318241964970404, 0.20160719073990638, -0...
                             ...                        
406    [0.06254474392219478, 0.021659289376740228, -0...
407    [0.09934862958419834, 0.0688907834303522, 0.05...
408    [-0.07860067645196228, -0.05415158554287136, 0...
409    [-0.014753283453705076, -0.08255328345370506, ...
410    [-0.06127647053494027, -0.02984192508039482, 0...
Name: 数据, Length: 411, dtype: object

In [12]:
df['故障标签'].unique()

array(['B007', 'B014', 'B021', 'IR007', 'IR014', 'IR021', 'OR007',
       'OR014', 'OR021', 'B028', 'IR028', 'Normal'], dtype=object)

In [7]:
df['rpm'].min()
# df['rpm'].max()
print(df.columns)
df['数据长度'].describe()

Index(['数据', '采样频率', '故障类型', '故障尺寸', '轴承位置', '采样位置', 'rpm', '载荷', '文件名',
       'or采样位置', '编号', '故障标签', '数据长度'],
      dtype='object')


count       411.000000
mean     123538.982968
std       51996.228154
min       63788.000000
25%       96000.000000
50%       96000.000000
75%      130549.000000
max      384000.000000
Name: 数据长度, dtype: float64

In [13]:
df[df['数据长度']==384000.000000]

,数据,采样频率,故障类型,故障尺寸,轴承位置,采样位置,rpm,载荷,文件名,or采样位置,编号,故障标签,数据长度
405,"[0.06483220523884395, 0.05940820523884394, 0.0...",48,Normal,None,Normal,DE,1772,1,N_1_(1772rpm).mat,None,X098,Normal,384000
406,"[0.06254474392219478, 0.021659289376740228, -0...",48,Normal,None,Normal,FE,1772,1,N_1_(1772rpm).mat,None,X098,Normal,384000
407,"[0.09934862958419834, 0.0688907834303522, 0.05...",48,Normal,None,Normal,DE,1750,2,N_2_(1750rpm).mat,None,X099,Normal,384000
408,"[-0.07860067645196228, -0.05415158554287136, 0...",48,Normal,None,Normal,FE,1750,2,N_2_(1750rpm).mat,None,X099,Normal,384000
409,"[-0.014753283453705076, -0.08255328345370506, ...",48,Normal,None,Normal,DE,1725,3,N_3.mat,None,X100,Normal,384000
410,"[-0.06127647053494027, -0.02984192508039482, 0...",48,Normal,None,Normal,FE,1725,3,N_3.mat,None,X100,Normal,384000


In [15]:
df['编号'].nunique()
import torch
print("CUDA版本:", torch.version.cuda)

CUDA版本: 12.8
